In [1]:
import numpy as np
from pytreenet.operators import Hamiltonian, TensorProduct
from pytreenet.ttno.state_diagram import TTNOFinder
from pytreenet.ttno.ttno_class import TreeTensorNetworkOperator
from fractions import Fraction
from TreeTopologies import random_impurity_binary_tree_ttn

Parameters. Same parameters for up and down spin.

In [ ]:
angle = 0
eshift = 0
Nb = 4
m = 16
U = 1.5

# hopping terms
hop = np.ones(Nb)

# bath on-site energy
eb = np.ones(Nb)

# impurity on-site energy
ei = -U / 2

state = random_impurity_binary_tree_ttn(Nb=Nb, m=m)

The SIAM Hamiltonian:
$$\hat H = U\,\hat n_{0\uparrow}\hat n_{0\downarrow} -\mu \sum_{\sigma} \hat C^{\dagger}_{0\sigma}\hat C_{0\sigma} + \sum_{i=1}^{N}\sum_{\sigma} \varepsilon_i\,
        \hat C^{\dagger}_{i\sigma}\hat C_{i\sigma} + t_i\,\hat C^{\dagger}_{0\sigma}\hat C_{i\sigma}
      + t_i^{*}\,\hat C^{\dagger}_{i\sigma}\hat C_{0\sigma}.$$

The site interaction are represented by lines and either represent hopping terms or for the impurity site a coulomb-like interaction.

<img src="./images/SIAM.png" width="500">

To reduce the order of the tensors, we introduce auxiliary tensors that do not represent physical sites.

<img src="./images/ttn_binary_tree.png" width="500">

Create empty hamiltonian to which we gonna add each term of the hamiltonian

In [3]:
hamiltonian = Hamiltonian()

Setting up the operators, the pauli-z we need to implement fermionic (orthogonal) anticommutation. 
The identities are required since they will act on the sites (physical and non-physical), on which no operator acts.
Or in different words the identities are used to fill the operator string. 
The non-physical tensors have a physical leg of dimension 1.

In [4]:
c = np.array([[0, 1], [0, 0]])
n = np.array([[0, 0], [0, 1]])
pz = np.array([[1, 0], [0, -1]])
conversion_dict = {}
conversion_dict["I2"] = np.eye(2)
conversion_dict["I1"] = np.eye(1)
conversion_dict["c"] = c
conversion_dict["c*"] = c.T
conversion_dict["n"] = n
conversion_dict["pz"] = pz

I just keep frac at 1. The coefficient map will be used to store the coefficient of each term.

In [5]:
frac = Fraction(1)
coeff_map = dict()

For time evolution at an angle, one can just multiply the hamiltonian with an angle-factor.
ATTENTION: angle between 0 and pi/2. One develops the state backwards in imaginary time -> High energy states are amplified.

$$\exp(-i(\cos(\phi)+i\sin(\phi))\hat Ht)$$

In [6]:
angle_fact = np.cos(angle) + 1j * np.sin(angle)
angle_fact = np.real_if_close(angle_fact)
angle_fact = -1j*(np.real_if_close(1j*(angle_fact)))

Include energy shift term, typically by the groundstate energy.
First term we are gonna iclude but not part of the Hamiltonian.

$$\exp(-i(\hat H - E_{0})t)$$

In [7]:
e_shift_term = TensorProduct()
e_shift_coeff_id = "E"
coeff_map[e_shift_coeff_id] = eshift * angle_fact
hamiltonian.add_term((frac, e_shift_coeff_id, e_shift_term))

Construct all terms including an operator acting on a bath site.
The Assumed order for the Jordan-Wigner Transformation is:
child0(up)|...|child{Nb-1}(up)|ImpUp(root)|ImpDown|child0(down)|...|child{Nb-1}(down)

In [8]:
for i in range(Nb):

        pzs_up = {f"child{j}(up)": "pz" for j in range(i + 1, Nb)}

        # Hopping term from impurity-site to bath-site i for up-spin.
        hop_term_up = TensorProduct(
            {f"child{i}(up)": "c*"} | pzs_up | {"ImpUp(root)": "c"}
        )
        hop_coeff_id = f"t{i}*"
        coeff_map[hop_coeff_id] = hop[i] * angle_fact
        hamiltonian.add_term((frac, hop_coeff_id, hop_term_up))

        # Hopping term bath-site i to impurity-site for up-spin.
        hop_term_up_dag = TensorProduct(
            {f"child{i}(up)": "c"} | pzs_up | {"ImpUp(root)": "c*"}
        )
        hop_conj_coeff_id = f"t{i}"
        coeff_map[hop_conj_coeff_id] = np.conj(hop[i]) * angle_fact
        hamiltonian.add_term((frac, hop_conj_coeff_id, hop_term_up_dag))

        # On-site energy term for bath-site i.
        os_term_up = TensorProduct({f"child{i}(up)": "n"})
        os_coeff_id = f"e{i}"
        coeff_map[os_coeff_id] = eb[i] * angle_fact
        hamiltonian.add_term((frac, os_coeff_id, os_term_up))

        ######
        # Same terms, but for spin down. 
        # Jordan-Wigner order differs though.
        pzs_down = {f"child{j}(down)": "pz" for j in range(0, i)}


        hop_term_down = TensorProduct(
            {"ImpDown": "c"} | pzs_down | {f"child{i}(down)": "c*"}
        )
        hamiltonian.add_term((frac, hop_coeff_id, hop_term_down))


        hop_term_down_dag = TensorProduct(
            {"ImpDown": "c*"} | pzs_down | {f"child{i}(down)": "c"}
        )
        hamiltonian.add_term((frac, hop_conj_coeff_id, hop_term_down_dag))


        os_term_down = TensorProduct({f"child{i}(down)": "n"})
        hamiltonian.add_term((frac, os_coeff_id, os_term_down))

On-site energy terms for the Impurity site

In [9]:
# both terms have the same coefficient (chemical potential)
os_imp_coeff_id = "eimp"
coeff_map[os_imp_coeff_id] = ei * angle_fact

# up-spin
os_term_up = TensorProduct({"ImpUp(root)": "n"})
hamiltonian.add_term((frac, os_imp_coeff_id, os_term_up))

# down-spin
os_term_down = TensorProduct({"ImpDown": "n"})
hamiltonian.add_term((frac, os_imp_coeff_id, os_term_down))

Interaction-term

In [10]:
inter_term = TensorProduct({"ImpUp(root)": "n", "ImpDown": "n"})
inter_coeff_id = "U"
coeff_map["U"] = U * angle_fact
hamiltonian.add_term((frac, inter_coeff_id, inter_term))

Creating the TTNO, the reference_tree gives the TTN structure, here two connected binary trees.

In [11]:
hamiltonian.conversion_dictionary = conversion_dict
hamiltonian.coeffs_mapping = coeff_map

ttno_ham = TreeTensorNetworkOperator.from_hamiltonian(
    hamiltonian=hamiltonian, reference_tree=state, method=TTNOFinder.SGE
)